In [2]:
import sqlite3
import pandas as pd

In [3]:
con = sqlite3.connect("shop.sqlite3")
cur = con.cursor()
print("Connected to shop.sqlite3")

Connected to shop.sqlite3


In [4]:
cur.executescript('''
DROP TABLE IF EXISTS order_items;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers;

CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    city TEXT NOT NULL,
    signup_date TEXT NOT NULL
);

CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    category TEXT NOT NULL,
    price REAL NOT NULL
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

CREATE TABLE order_items (
    order_item_id INTEGER PRIMARY KEY,
    order_id INTEGER NOT NULL,
    product_id INTEGER NOT NULL,
    quantity INTEGER NOT NULL,
    FOREIGN KEY (order_id) REFERENCES orders(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
''')
print("Tables created")

Tables created


In [5]:
customers = [
    (1, "Ram Shrestha", "Kathmandu", "2024-01-15"),
    (2, "Sita Gurung", "Pokhara", "2024-02-03"),
    (3, "Hari Tamang", "Biratnagar", "2024-02-20"),
    (4, "Gita Rai", "Kathmandu", "2024-03-05"),
    (5, "Mina Thapa", "Lalitpur", "2024-03-18"),
    (6, "Suresh Karki", "Pokhara", "2024-04-02"),
    (7, "Anita Magar", "Bhaktapur", "2024-04-22"),
    (8, "Bikash Limbu", "Biratnagar", "2024-05-10"),
]

products = [
    (1, "Wireless Mouse", "Electronics", 12.50),
    (2, "Mechanical Keyboard", "Electronics", 45.00),
    (3, "Notebook", "Stationery", 2.50),
    (4, "Backpack", "Accessories", 30.00),
    (5, "Water Bottle", "Accessories", 8.00),
    (6, "USB-C Cable", "Electronics", 6.00),
    (7, "Desk Lamp", "Home", 18.00),
    (8, "Pen Set", "Stationery", 4.00),
]

orders = [
    (1, 1, "2024-05-01"), (2, 2, "2024-05-03"), (3, 1, "2024-05-10"),
    (4, 3, "2024-05-12"), (5, 4, "2024-05-15"), (6, 2, "2024-05-20"),
    (7, 5, "2024-05-22"), (8, 6, "2024-05-25"), (9, 1, "2024-06-01"),
    (10, 7, "2024-06-03"), (11, 4, "2024-06-05"), (12, 8, "2024-06-10"),
]

order_items = [
    (1, 1, 1, 2), (2, 1, 6, 1), (3, 2, 4, 1), (4, 3, 2, 1),
    (5, 4, 3, 5), (6, 4, 8, 3), (7, 5, 7, 1), (8, 6, 1, 1),
    (9, 6, 5, 2), (10, 7, 4, 1), (11, 7, 6, 2), (12, 8, 2, 1),
    (13, 9, 3, 10), (14, 9, 5, 1), (15, 10, 7, 2), (16, 11, 1, 3),
    (17, 12, 4, 1), (18, 12, 8, 2),
]

cur.executemany("INSERT INTO customers VALUES (?,?,?,?)", customers)
cur.executemany("INSERT INTO products VALUES (?,?,?,?)", products)
cur.executemany("INSERT INTO orders VALUES (?,?,?)", orders)
cur.executemany("INSERT INTO order_items VALUES (?,?,?,?)", order_items)
con.commit()
print("Sample data inserted")

Sample data inserted


In [10]:
def run(query):
    return pd.read_sql_query(query, con)

run("SELECT * FROM customers ;")

,customer_id,name,city,signup_date
0,1,Ram Shrestha,Kathmandu,2024-01-15
1,2,Sita Gurung,Pokhara,2024-02-03
2,3,Hari Tamang,Biratnagar,2024-02-20
3,4,Gita Rai,Kathmandu,2024-03-05
4,5,Mina Thapa,Lalitpur,2024-03-18
5,6,Suresh Karki,Pokhara,2024-04-02
6,7,Anita Magar,Bhaktapur,2024-04-22
7,8,Bikash Limbu,Biratnagar,2024-05-10


In [11]:
run('''
SELECT customer_id, name, city, signup_date
FROM customers
WHERE city = 'Kathmandu';
''')

,customer_id,name,city,signup_date
0,1,Ram Shrestha,Kathmandu,2024-01-15
1,4,Gita Rai,Kathmandu,2024-03-05


In [12]:

run('''
SELECT product_name, category, price
FROM products
WHERE category = 'Electronics'
ORDER BY price ASC;
''')

,product_name,category,price
0,USB-C Cable,Electronics,6.0
1,Wireless Mouse,Electronics,12.5
2,Mechanical Keyboard,Electronics,45.0


In [13]:
run('''
SELECT name, city, signup_date
FROM customers
WHERE signup_date > '2024-03-01'
ORDER BY signup_date DESC;
''')

,name,city,signup_date
0,Bikash Limbu,Biratnagar,2024-05-10
1,Anita Magar,Bhaktapur,2024-04-22
2,Suresh Karki,Pokhara,2024-04-02
3,Mina Thapa,Lalitpur,2024-03-18
4,Gita Rai,Kathmandu,2024-03-05


In [14]:
run('''
SELECT city, COUNT(*) AS num_customers
FROM customers
GROUP BY city
ORDER BY num_customers DESC;
''')

,city,num_customers
0,Pokhara,2
1,Kathmandu,2
2,Biratnagar,2
3,Lalitpur,1
4,Bhaktapur,1


In [15]:
run('''
SELECT category,
       MIN(price) AS cheapest,
       MAX(price) AS priciest,
       ROUND(AVG(price), 2) AS avg_price
FROM products
GROUP BY category
ORDER BY avg_price DESC;
''')

,category,cheapest,priciest,avg_price
0,Electronics,6.0,45.0,21.17
1,Accessories,8.0,30.0,19.00
2,Home,18.0,18.0,18.00
3,Stationery,2.5,4.0,3.25


In [16]:
run('''
SELECT customer_id, COUNT(*) AS order_count
FROM orders
GROUP BY customer_id
ORDER BY order_count DESC;
''')

,customer_id,order_count
0,1,3
1,4,2
2,2,2
3,8,1
4,7,1
5,6,1
6,5,1
7,3,1


In [17]:
run("SELECT COUNT(*) AS total_orders FROM orders;")

,total_orders
0,12


In [18]:
run('''
SELECT o.order_id, c.name AS customer_name, o.order_date
FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id
ORDER BY o.order_date;
''')

,order_id,customer_name,order_date
0,1,Ram Shrestha,2024-05-01
1,2,Sita Gurung,2024-05-03
2,3,Ram Shrestha,2024-05-10
3,4,Hari Tamang,2024-05-12
4,5,Gita Rai,2024-05-15
5,6,Sita Gurung,2024-05-20
6,7,Mina Thapa,2024-05-22
7,8,Suresh Karki,2024-05-25
8,9,Ram Shrestha,2024-06-01
9,10,Anita Magar,2024-06-03


In [19]:
run('''
SELECT c.name, COUNT(o.order_id) AS order_count
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.name
ORDER BY order_count ASC;
''')

,name,order_count
0,Hari Tamang,1
1,Mina Thapa,1
2,Suresh Karki,1
3,Anita Magar,1
4,Bikash Limbu,1
5,Sita Gurung,2
6,Gita Rai,2
7,Ram Shrestha,3


In [20]:
run('''
SELECT oi.order_id, p.product_name, p.category, oi.quantity, p.price,
       ROUND(oi.quantity * p.price, 2) AS line_total
FROM order_items oi
INNER JOIN products p ON oi.product_id = p.product_id
ORDER BY oi.order_id;
''')

,order_id,product_name,category,quantity,price,line_total
0,1,Wireless Mouse,Electronics,2,12.5,25.0
1,1,USB-C Cable,Electronics,1,6.0,6.0
2,2,Backpack,Accessories,1,30.0,30.0
3,3,Mechanical Keyboard,Electronics,1,45.0,45.0
4,4,Notebook,Stationery,5,2.5,12.5
5,4,Pen Set,Stationery,3,4.0,12.0
6,5,Desk Lamp,Home,1,18.0,18.0
7,6,Wireless Mouse,Electronics,1,12.5,12.5
8,6,Water Bottle,Accessories,2,8.0,16.0
9,7,Backpack,Accessories,1,30.0,30.0


In [21]:
run('''
SELECT p.product_name,
       SUM(oi.quantity) AS units_sold,
       ROUND(SUM(oi.quantity * p.price), 2) AS total_revenue
FROM order_items oi
INNER JOIN products p ON oi.product_id = p.product_id
GROUP BY p.product_id, p.product_name
ORDER BY total_revenue DESC;
''')

,product_name,units_sold,total_revenue
0,Mechanical Keyboard,2,90.0
1,Backpack,3,90.0
2,Wireless Mouse,6,75.0
3,Desk Lamp,3,54.0
4,Notebook,15,37.5
5,Water Bottle,3,24.0
6,Pen Set,5,20.0
7,USB-C Cable,3,18.0


In [22]:
run('''
SELECT p.category,
       SUM(oi.quantity) AS units_sold,
       ROUND(SUM(oi.quantity * p.price), 2) AS total_revenue
FROM order_items oi
INNER JOIN products p ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY total_revenue DESC;
''')

,category,units_sold,total_revenue
0,Electronics,11,183.0
1,Accessories,6,114.0
2,Stationery,20,57.5
3,Home,3,54.0


In [23]:
run('''
SELECT c.name,
       COUNT(DISTINCT o.order_id) AS num_orders,
       ROUND(SUM(oi.quantity * p.price), 2) AS total_spent,
       ROUND(SUM(oi.quantity * p.price) * 1.0 / COUNT(DISTINCT o.order_id), 2) AS avg_order_value
FROM customers c
INNER JOIN orders o ON c.customer_id = o.customer_id
INNER JOIN order_items oi ON o.order_id = oi.order_id
INNER JOIN products p ON oi.product_id = p.product_id
GROUP BY c.customer_id, c.name
ORDER BY total_spent DESC;
''')

,name,num_orders,total_spent,avg_order_value
0,Ram Shrestha,3,109.0,36.33
1,Sita Gurung,2,58.5,29.25
2,Gita Rai,2,55.5,27.75
3,Suresh Karki,1,45.0,45.00
4,Mina Thapa,1,42.0,42.00
5,Bikash Limbu,1,38.0,38.00
6,Anita Magar,1,36.0,36.00
7,Hari Tamang,1,24.5,24.50


In [24]:
run('''
SELECT c.name, ROUND(SUM(oi.quantity * p.price), 2) AS total_spent
FROM customers c
INNER JOIN orders o ON c.customer_id = o.customer_id
INNER JOIN order_items oi ON o.order_id = oi.order_id
INNER JOIN products p ON oi.product_id = p.product_id
GROUP BY c.customer_id, c.name
HAVING SUM(oi.quantity * p.price) > 50
ORDER BY total_spent DESC;
''')

,name,total_spent
0,Ram Shrestha,109.0
1,Sita Gurung,58.5
2,Gita Rai,55.5


In [25]:
run('''
SELECT p.product_name, ROUND(SUM(oi.quantity * p.price), 2) AS total_revenue
FROM order_items oi
INNER JOIN products p ON oi.product_id = p.product_id
GROUP BY p.product_id, p.product_name
ORDER BY total_revenue DESC
LIMIT 1;
''')

,product_name,total_revenue
0,Mechanical Keyboard,90.0


In [26]:
con.close()
print("Connection closed.")

Connection closed.
